https://github.com/ijstokes/tutorial-local-llm

---

# Local LLM

---

In [21]:
# load in libraries
import json
from ollama import Client, chat
from pathlib import Path
import pandas as pd

In [22]:
import subprocess
import time

# Start the Ollama server if it is not already running
ollama_server = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

# Give the server a moment to start
time.sleep(3)

# Check whether the server is running
print("Ollama process:", ollama_server.poll())

Ollama process: None


In [23]:
from ollama import list as list_models

# Check which models are available
models = list_models()

print(models)

models=[Model(model='llama3.2:latest', modified_at=datetime.datetime(2026, 10, 27, 7, 43, 57, 137259, tzinfo=TzInfo(39600)), digest='a80c4f17acd55265feec403c7aef86be0c25983ab279d83f3bcd3abbcb5b8b72', size=2019393189, details=ModelDetails(parent_model='', format='gguf', family='llama', families=['llama'], parameter_size='3.2B', quantization_level='Q4_K_M')), Model(model='codellama:latest', modified_at=datetime.datetime(2026, 10, 27, 7, 43, 51, 300149, tzinfo=TzInfo(39600)), digest='8fdf8f752f6e80de33e82f381aba784c025982752cd1ae9377add66449d2225f', size=3825910662, details=ModelDetails(parent_model='', format='gguf', family='llama', families=None, parameter_size='7B', quantization_level='Q4_0')), Model(model='mistral:latest', modified_at=datetime.datetime(2026, 10, 27, 7, 43, 46, 29050, tzinfo=TzInfo(39600)), digest='6577803aa9a036369e481d648a2baebb381ebc6e897f2bb9a766a2aa7bfbc1cf', size=4372824384, details=ModelDetails(parent_model='', format='gguf', family='llama', families=['llama'], 

In [62]:
CONDITION = "aig"

# Set the path to the processed passages
DATA_DIR = Path("..") / "pmc" / CONDITION / "processed"

# Load the diagnostic passages grouped by article
diagnostic_articles = pd.read_parquet(
    DATA_DIR / "diagnostic_articles.parquet"
)

# Check the number of articles and passages
print("Articles:", len(diagnostic_articles))
print("Passages:", diagnostic_articles["n_passages"].sum())

display(diagnostic_articles)

Articles: 5
Passages: 10


,pmcid,text,passage_ids,n_passages,max_score
0,PMC11010983,The prevalence of AAG in the general populatio...,"[5, 17]",2,69.326569
1,PMC11863545,The objective of this retrospective cohort stu...,[6],1,69.230820
2,PMC3520153,The cut-off values adopted were those suggeste...,[8],1,69.347687
3,PMC6502205,"In practical terms, it is advisable to screen ...",[16],1,69.181526
4,PMC8621657,CAG can be asymptomatic or can be characterize...,"[9, 20, 26, 27, 28]",5,70.686111


In [63]:
# Connect to the local Ollama server
client = Client(host="http://localhost:11434")

# select the local llm to use
MODEL = "llama3.2"

In [69]:
# Create instructions for extracting diagnostic criteria
PROMPT = """
Extract diagnostic criteria from the following medical passage.

Only extract information explicitly stated in the passage.
Do not invent diagnostic criteria or thresholds.
Distinguish diagnostic criteria from symptoms, complications
and associated findings.

Return a JSON object containing a list named "criteria".

For each diagnostic criterion, extract:
- condition: the disease being diagnosed
- criterion: description of the diagnostic criterion
- category: type of diagnostic criterion
- diagnostic_test: test used to identify the criterion
- marker: biomarker or characteristic being measured
- threshold: numerical cut-off including comparison operator
- unit: measurement unit
- diagnostic_role: confirmatory, supportive, exclusionary or unspecified
- combination: other criteria required in combination
- evidence: exact supporting text from the passage

Use null when information is not explicitly stated.
Preserve numerical thresholds and combined criteria.
Ensure the evidence directly supports the extracted criterion.

If no diagnostic criteria are found, return:
{{"criteria": []}}

Passage:
{passage}
"""

In [70]:
prompt = PROMPT

In [71]:
# Create a function to send prompts to the local LLM
def run_llm(prompt, model=MODEL):

    # Send the prompt to the local LLM
    response = chat(
        model=model,
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        options={
            "temperature": 0,
            "num_ctx": 16384
        }
    )

    # Return the generated response
    return response.message.content, response.done_reason

In [72]:
# Create a list to store the extraction results
results = []

# Go through each article
for _, row in diagnostic_articles.iterrows():

    PMCID = row["pmcid"]
    passage = row["text"]

    # Insert the grouped passages into the diagnostic prompt
    prompt = PROMPT.format(passage=passage)

    # Extract diagnostic criteria
    result = run_llm(prompt)

    # Store the extraction results
    results.append({
        "pmcid": PMCID,
        "model": MODEL,
        "retrieval_method": "medcpt",
        "n_passages": int(row["n_passages"]),
        "passage_ids": list(row["passage_ids"]),
        "response": result
    })

    print("Processed:", PMCID)

Processed: PMC11010983
Processed: PMC11863545
Processed: PMC3520153
Processed: PMC6502205
Processed: PMC8621657


In [74]:
# Convert passage IDs to Python integers
for result in results:
    result["passage_ids"] = [
        int(pid) for pid in result["passage_ids"]
    ]

In [75]:
# Save the diagnostic extraction results
with open(
    DATA_DIR / "medcpt_llm_results.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(results, f, indent=4, ensure_ascii=False)

In [76]:
# Convert the results into a DataFrame
results_df = pd.DataFrame(results)

display(results_df)

,pmcid,model,retrieval_method,n_passages,passage_ids,response
0,PMC11010983,llama3.2,medcpt,2,"[5, 17]","({""criteria"": [\n {\n ""condition"": ""At..."
1,PMC11863545,llama3.2,medcpt,1,[6],"({""criteria"": [\n {\n ""condition"": ""Autoim..."
2,PMC3520153,llama3.2,medcpt,1,[8],(Since the passage does not explicitly state a...
3,PMC6502205,llama3.2,medcpt,1,[16],"({""criteria"": [\n {\n ""condition"": ""Ch..."
4,PMC8621657,llama3.2,medcpt,5,"[9, 20, 26, 27, 28]","({""criteria"": [\n {\n ""condition"": ""Ch..."


In [77]:
# Display the extracted diagnostic criteria for each article
for result in results:
    print("Article:", result["pmcid"])
    print(result["response"])
    print("-" * 80)

Article: PMC11010983
('{"criteria": [\n    {\n      "condition": "Atrophic Gastritis",\n      "criterion": "Mucosal atrophy",\n      "category": "Endoscopic finding",\n      "diagnostic_test": "Endoscopy",\n      "marker": "absence of micro-vessels",\n      "threshold": null,\n      "unit": "absence",\n      "diagnostic_role": "supportive",\n      "combination": null,\n      "evidence": "In normal conditions, micro-vessels of the gastric mucosa of the body show a honeycomb subepithelial capillary network and a regular-shape venous collection, which are absent in patients with atrophic gastritis [37]."\n    },\n    {\n      "condition": "Atrophic Gastritis",\n      "criterion": "Mucosal erythema",\n      "category": "Endoscopic finding",\n      "diagnostic_test": "Endoscopy",\n      "marker": "presence of mucosal erythema",\n      "threshold": null,\n      "unit": "presence",\n      "diagnostic_role": "supportive",\n      "combination": null,\n      "evidence": "Endoscopic findings in A

In [79]:
# Display the extracted diagnostic criteria
for result in results:

    print("Article:", result["pmcid"])
    print(result["response"][0])
    print("-" * 80)

Article: PMC11010983
{"criteria": [
    {
      "condition": "Atrophic Gastritis",
      "criterion": "Mucosal atrophy",
      "category": "Endoscopic finding",
      "diagnostic_test": "Endoscopy",
      "marker": "absence of micro-vessels",
      "threshold": null,
      "unit": "absence",
      "diagnostic_role": "supportive",
      "combination": null,
      "evidence": "In normal conditions, micro-vessels of the gastric mucosa of the body show a honeycomb subepithelial capillary network and a regular-shape venous collection, which are absent in patients with atrophic gastritis [37]."
    },
    {
      "condition": "Atrophic Gastritis",
      "criterion": "Mucosal erythema",
      "category": "Endoscopic finding",
      "diagnostic_test": "Endoscopy",
      "marker": "presence of mucosal erythema",
      "threshold": null,
      "unit": "presence",
      "diagnostic_role": "supportive",
      "combination": null,
      "evidence": "Endoscopic findings in AAG include mucosal erythe

In [78]:
# Stop the Ollama server
ollama_server.terminate()
ollama_server.wait()


0